In [24]:
from langgraph.graph import StateGraph,START,END
from langchain_core.messages import BaseMessage,HumanMessage
from typing import TypedDict,Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [25]:
load_dotenv()

True

In [26]:
## initalize llm
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

In [27]:
from langgraph.graph.message import add_messages

class chatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]


In [28]:
## state of graph

checkpoint = MemorySaver()

graph = StateGraph(chatState)

In [29]:
## chatbot node function

def chatbot(state:chatState):
    message = state['messages']

    response = llm.invoke(message)
    return {'messages': [response]}

In [30]:
## nodes

graph.add_node("chatbot",chatbot)

In [31]:
##edges
graph.add_edge(START,"chatbot")
graph.add_edge("chatbot",END)

In [32]:
# initial_state = {
#     'messages' : [HumanMessage("what is object oriented programming")]
# }

workflow = graph.compile(checkpointer=checkpoint)

In [35]:
thread_id = "1"


while True:
    user_messages = input("Type here: ")
    print("user" , user_messages)

    if user_messages.strip().lower() in ["quit","end","bye"]:
        break
    config = {'configurable':{'thread_id':thread_id}}
    response = workflow.invoke({
        'messages' : [HumanMessage(content=user_messages)]
    },config=config)

    print("AI:", response['messages'][-1].content)    


user what is  my name
AI: Your name is Zain.
user quit
